# 01 · ROME on modern LLMs (§4): Tables 1 and 2

This notebook recomputes everything the paper reports about the ROME edits themselves:

| Paper item | What is computed |
|---|---|
| §4.1, *Main evaluation* | the same 1,000 CounterFact cases on all 17 models, ten model families |
| §4.1, *Layer selection* | the edit layer $\ell^*$ lies in the first half of every network |
| Table 2 | $\ell^*/L$, S, ES, PS, NS for the 17 models |
| Table 1 | the four reference models against the values reported by Meng et al. (2022) |
| §4.2, Abstract, Introduction, Conclusion | ES ≥ 90% on all models and ≥ 97% on 15, the 2026 models, score ranges |
| §5.3, §5.6 | 345 of the 17,000 edits did not reach ES = 1 |

**Inputs:** `existing-renderer/classic-gram/<model>/summary.json`, `data/classic-family-summary.csv`, `manifests/classic-gram/rome-facts-per-model.csv`.
Each check at the end compares a computed value with the value printed in the paper.

In [ ]:
import pandas as pd
from paper_common import (MODELS, EXCLUDED, OUT, read_csv, base_model_results, inputs_used,
                          Checks, latex_rows_equal, diff_rows)

pd.set_option("display.precision", 2)
checks = Checks()

df = base_model_results()
df[["name", "target", "L", "S", "ES", "PS", "NS", "n", "rome_success"]]

## The same 1,000 CounterFact edits on every model

Every model receives the same 1,000 cases from CounterFact, sampled once with a fixed seed (§4.1).
The archive also contains Gemma 4 12B, which was removed from the study; it is dropped here.

In [ ]:
facts = read_csv("manifests/classic-gram/rome-facts-per-model.csv")
facts = facts[~facts.family.isin(EXCLUDED)]
case_sets = facts.groupby("family").case_id.apply(frozenset)

checks.value("models in the study", "§4.1, Abstract", case_sets.size, 17)
checks.value("model families", "Abstract, §1", MODELS.family.nunique(), 10)
checks.value("distinct case sets across models (1 = identical)", "§4.1", case_sets.nunique(), 1)
checks.value("CounterFact edits per model", "§4.1", len(case_sets.iloc[0]), 1000)
checks.value("total edits", "§5.3, Abstract", df.n.sum(), 17000)

layers = facts.groupby("family").configured_rome_layer.unique().apply(lambda v: int(v[0]))
checks.same("edit layer in the fact manifest equals l* of Table 2", "§4.1", layers.to_dict(), df.set_index("key").target.to_dict())
checks.same("l* lies in the first half of every network", "§4.1", bool((df.target < df.L / 2).all()), True)
pd.DataFrame({"cases": case_sets.apply(len), "l*": layers})

## Table 2: ROME efficacy on the 17 models

ES, PS and NS are the mean CounterFact efficacy, paraphrase and neighborhood scores over the 1,000 edits, in percent.
S is their harmonic mean. $\ell^*/L$ is the edited layer (0-indexed) and the number of layers.
The rows are grouped by release period as in the paper, and written to `outputs/table2_rows.tex`.

In [ ]:
def table2_rows(d):
    lines = []
    for i, (_, g) in enumerate(d.groupby("period", sort=False)):
        if i:
            lines.append(r"\addlinespace")
        for r in g.itertuples():
            lines.append(f"{r.name:<22} & {r.target}/{r.L} & {r.S:5.1f} & {r.ES:5.1f} & {r.PS:5.1f} & {r.NS:5.1f} \\\\")
    return "\n".join(lines)

PAPER_TABLE2 = r"""
GPT-2 Medium           & 8/24 &  86.2 &  98.0 &  95.8 &  70.7 \\
GPT-2 Large            & 12/36 &  86.7 &  98.0 &  93.7 &  72.9 \\
GPT-2 XL               & 16/48 &  87.9 &  98.1 &  94.2 &  75.0 \\
GPT-J-6B               & 5/28 &  90.3 &  99.8 &  98.2 &  76.8 \\
\addlinespace
OPT-6.7B               & 14/32 &  91.2 &  98.8 &  95.3 &  81.5 \\
Falcon-7B              & 6/32 &  92.5 &  98.9 &  96.8 &  83.5 \\
Llama-2-7B             & 5/32 &  89.6 &  97.6 &  92.5 &  80.4 \\
Mistral-7B v0.1        & 7/32 &  85.6 &  93.1 &  83.4 &  81.4 \\
DeepSeek-LLM-7B Base   & 6/30 &  91.5 &  99.0 &  97.9 &  80.2 \\
\addlinespace
Mistral-7B v0.3        & 7/32 &  83.1 &  90.5 &  78.4 &  81.5 \\
Qwen3-4B               & 8/36 &  90.2 &  99.5 &  97.0 &  77.6 \\
Qwen3-8B               & 10/36 &  89.3 &  99.8 &  98.7 &  74.3 \\
Granite~4.0 Micro      & 10/40 &  90.2 &  97.1 &  97.9 &  78.5 \\
Olmo~3 7B              & 6/32 &  91.3 &  99.9 &  97.0 &  79.7 \\
Ministral~3 8B         & 7/34 &  91.8 &  99.7 &  97.5 &  80.6 \\
\addlinespace
Qwen3.5-4B             & 5/32 &  90.6 &  99.6 &  98.7 &  77.2 \\
Granite~4.1 8B         & 13/40 &  92.2 &  98.1 &  98.0 &  82.5 \\
"""

rows2 = table2_rows(df)
(OUT / "table2_rows.tex").write_text(rows2 + "\n")
checks.same("Table 2 rows equal the paper", "Table 2", latex_rows_equal(rows2, PAPER_TABLE2), True)
print(rows2)
diff_rows(rows2, PAPER_TABLE2)

## Table 1: reproduction on the four models of the original ROME evaluation

*Orig* are the values reported by Meng et al. (2022); *Ours* are the corresponding rows of Table 2.
The paper states that the reimplementation is within 2.6 points of the reported values on every metric and model.

In [ ]:
ORIG = pd.DataFrame(
    {"S": [87.4, 88.2, 89.2, 91.5], "ES": [100.0, 99.9, 100.0, 99.9],
     "PS": [96.4, 96.3, 96.4, 99.1], "NS": [71.8, 73.4, 75.4, 78.9]},
    index=["gpt2-medium", "gpt2-large", "gpt2-xl", "gpt-j-6b"])
PAPER_OURS = {  # Table 1, rows "Ours": S, ES, PS, NS
    "gpt2-medium": ("86.2", "98.0", "95.8", "70.7"), "gpt2-large": ("86.7", "98.0", "93.7", "72.9"),
    "gpt2-xl": ("87.9", "98.1", "94.2", "75.0"), "gpt-j-6b": ("90.3", "99.8", "98.2", "76.8")}

ours = df.set_index("key").loc[ORIG.index, ["S", "ES", "PS", "NS"]]
for key, printed in PAPER_OURS.items():
    for metric, value in zip(["S", "ES", "PS", "NS"], printed):
        checks.printed(f"Table 1, {key}, {metric} (Ours)", "Table 1", ours.loc[key, metric], value)
gap = (ORIG - ours.round(1)).abs()
checks.printed("largest gap to the reported values (points)", "§4.1, Reproduction", gap.values.max(), "2.6")
pd.concat({"Orig": ORIG, "Ours": ours.round(1), "|gap|": gap.round(1)}, axis=1)

## Statements in the text

These are the ROME numbers quoted in §4.2, the abstract, the introduction and the conclusion.

In [ ]:
d = df.set_index("name")
checks.printed("lowest ES of all 17 models (>= 90%)", "§4.2, Abstract, §1", df.ES.min(), "90.5")
checks.value("models with ES >= 97%", "§4.2, §1", int((df.ES.round(1) >= 97).sum()), 15)
checks.printed("ES of Qwen3.5-4B (2026)", "§4.2, §1", d.loc["Qwen3.5-4B", "ES"], "99.6")
checks.printed("ES of Granite 4.1 8B (2026)", "§4.2, §1", d.loc["Granite~4.1 8B", "ES"], "98.1")
checks.same("the two lowest ES are the two Mistral-7B models", "§4.2",
            set(df.nsmallest(2, "ES").name), {"Mistral-7B v0.1", "Mistral-7B v0.3"})
checks.printed("ES of Mistral-7B v0.1", "§4.2", d.loc["Mistral-7B v0.1", "ES"], "93.1")
checks.printed("ES of Mistral-7B v0.3", "§4.2", d.loc["Mistral-7B v0.3", "ES"], "90.5")
checks.printed("PS of Mistral-7B v0.1", "§4.2", d.loc["Mistral-7B v0.1", "PS"], "83.4")
checks.printed("PS of Mistral-7B v0.3", "§4.2", d.loc["Mistral-7B v0.3", "PS"], "78.4")
others = df[~df.name.str.startswith("Mistral-7B")]
checks.printed("lowest PS of every other model", "§4.2", others.PS.min(), "92.5")
checks.same("lowest NS model", "§4.2", df.loc[df.NS.idxmin(), "name"], "GPT-2 Medium")
checks.printed("lowest NS", "§4.2", df.NS.min(), "70.7")
checks.same("highest NS model", "§4.2", df.loc[df.NS.idxmax(), "name"], "Falcon-7B")
checks.printed("highest NS", "§4.2", df.NS.max(), "83.5")
checks.same("lowest S model", "§4.2", df.loc[df.S.idxmin(), "name"], "Mistral-7B v0.3")
checks.printed("lowest S", "§4.2", df.S.min(), "83.1")
checks.same("highest S model", "§4.2", df.loc[df.S.idxmax(), "name"], "Falcon-7B")
checks.printed("highest S", "§4.2", df.S.max(), "92.5")
multimodal = df[df.key.isin(["ministral-3-8b", "qwen3.5-4b"])]
checks.printed("lowest ES of the two multimodal models", "§4.3", multimodal.ES.min(), "99.6")
checks.value("edits that did not reach ES = 1", "§5.3, §5.6", (df.n - df.rome_success).sum(), 345)
checks.value("edits that reached ES = 1", "§5.3", df.rome_success.sum(), 16655)

## Summary of checks

In [ ]:
checks.table().to_csv(OUT / "checks_01_rome.csv", index=False)
checks.assert_all()
checks.table()

In [ ]:
inputs_used()